# Lab 03A: Dynamic Programming, Optimal Coin Change

Ashesi University | CS456 Algorithm Design & Analysis | Week 10


**Fill in your details in the box below.**


Name of group member 1: Blaise Izerimana

ID of group member 1: 48452028

Name of group member 2: Fida Ukwishaka

ID of group member 2: 47582028


## Instructions

- This notebook is **Lab A**. Finish it within **90 minutes**.
- Everything is inside this notebook. You do not download or upload anything and you do not use a terminal.
- Work alone or in a pair (maximum 2 per group). Use Python.
- **You write your own dynamic programming solution.** Name your variables and organise your code however you like. The only fixed things are the function name and what it returns (described below). Use loops (tabulation), not recursion.
- You will compare your DP with two ready-made methods: a **naive** method (tries everything, remembers nothing) and a **greedy** method (always takes the best-looking option now). Their code is in the Step 0 cell. Run it, do not change it, and do not use these methods inside your DP.
- You may use `normalize_coins` (removes duplicate denominations and sorts them) and `validate_amount` from the Step 0 cell. Do not change the list of coins you are given.
- Follow the course's published rules on permitted assistance, and acknowledge any external sources at the end of this notebook.
- Run the cells from top to bottom. After you write your DP, run the cells below it again to test it.

## Working independently or in pairs

One student drives while the other checks and tests; swap by minute 35. Both must be able to explain the state and reconstruct the example. Solo students complete the same core tasks; pairs do not need extra algorithms. Each person writes their own exit response.

## Use your 90 minutes

| Minutes | Required work                                                                         |
| ------- | ------------------------------------------------------------------------------------- |
| 0-12    | Define the state, recurrence, base cases and evaluation order; trace a small instance |
| 12-40   | Implement the DP value computation and count its work                                 |
| 40-55   | Reconstruct the solution; run the tests and add two of your own                       |
| 55-70   | Compare your DP with the naive and greedy methods                                     |
| 70-85   | Explain correctness, complexity and measured results; prepare your submission         |
| 85-90   | Each student writes an individual exit response                                       |

## What to submit

1. This completed notebook, with all cells run so the outputs are visible (this includes your tests).
2. The experiment CSV (the experiment cell saves it for you).
3. A short analysis of **at most 600 words** (your answers to the analysis questions).
4. Names and a two-line contribution statement.
5. An individual exit response of **at most 80 words** from each person.

## Marking

| Criterion                                               | Marks  |
| ------------------------------------------------------- | ------ |
| State, recurrence, boundaries and evaluation order      | 6      |
| Correct DP implementation and counting of work          | 8      |
| Valid optimal solution reconstruction                   | 5      |
| Tests including edge cases and greedy counterexample    | 4      |
| Correctness, complexity and experimental interpretation | 5      |
| Individual exit response                                | 2      |
| **Total**                                               | **30** |

The first 28 marks assess the submitted work; the last 2 are awarded individually. Extensions are optional, ungraded and not required for full marks.


## Step 0: Load the setup code

Run the next cell. It contains two small helpers and the naive and greedy methods you will compare against. **Do not change it.**

- `normalize_coins(coins)` checks the denominations, removes duplicates and sorts them. `validate_amount(amount)` checks that the amount is a nonnegative integer.
- `coin_naive(coins, amount)` tries every possible final coin with plain recursion and no memory. It returns the minimum number of coins and how many recursive calls it needed. It stops after 50,000 calls and then says `BUDGET_EXCEEDED`.
- `coin_greedy(coins, amount)` always takes the largest coin that fits. It returns the number of coins and the list of coins, or `None` if it gets stuck.


In [14]:
# Run this cell once. Do not change it.
# It holds two small helpers and the two non-DP methods you will compare your DP against.

CALL_LIMIT = 50000   # the naive method gives up after this many recursive calls

class BudgetExceeded(Exception):
    pass

def normalize_coins(coins):
    """Check the denominations, remove duplicates and sort them (smallest first)."""
    values = list(coins)
    if any(type(c) is not int or c <= 0 for c in values):
        raise ValueError("Denominations must be positive integers")
    return sorted(set(values))

def validate_amount(amount):
    """Check that the amount is a nonnegative integer."""
    if type(amount) is not int or amount < 0:
        raise ValueError("Amount must be a nonnegative integer")

def coin_naive(coins, amount):
    """Plain recursion with no memory: try every possible final coin.
    Returns (minimum number of coins or None, number of recursive calls).
    If it needs more than CALL_LIMIT calls it stops and returns "BUDGET_EXCEEDED"."""
    validate_amount(amount)
    coins = normalize_coins(coins)
    calls = 0

    def solve(remaining):
        nonlocal calls
        if calls >= CALL_LIMIT:
            raise BudgetExceeded()
        calls += 1
        if remaining == 0:
            return 0
        best = None
        for coin in coins:
            if coin <= remaining:
                sub = solve(remaining - coin)
                if sub is not None and (best is None or sub + 1 < best):
                    best = sub + 1
        return best

    try:
        return solve(amount), calls
    except BudgetExceeded:
        return "BUDGET_EXCEEDED", calls

def coin_greedy(coins, amount):
    """Always take the largest coin that fits, as many times as it fits.
    Returns (number of coins or None, list of coins). None means it got stuck."""
    validate_amount(amount)
    selected = []
    remaining = amount
    for coin in reversed(normalize_coins(coins)):
        while coin <= remaining:
            selected.append(coin)
            remaining -= coin
    if remaining > 0:
        return None, []
    return len(selected), selected


# Lab A: Optimal Coin Change

**Scenario.** A payment robot must produce an exact amount S using an unlimited supply of denominations D. Greedy choice is tempting, but the smallest number of coins is required. Denominations are positive integers and the target is a nonnegative integer.

**What your function must be**

`coin_change_dp(coins, amount)` must return three things: `count, selected, stats`

- `count`: the minimum number of coins. Return `None` if the amount cannot be made, and `0` for amount 0.
- `selected`: a list of coins that adds up to the amount exactly (coins may repeat). It has `count` coins. Use `[]` for an unreachable amount or amount 0.
- `stats`: a dictionary `{"states": ..., "checks": ...}` where **states** is the number of positive amounts your loop processes (1 up to the amount) and **checks** is how many times you check a denomination against an amount, **including coins that are too big**.
- If two final coins give the same best count, choose the **smallest** coin.
- Build fresh tables every time the function is called (nothing may be remembered between calls).


## Task 1: Design before coding


**Question a.** Define F(s) precisely.


Let **F(s)** be the minimum number of coins needed to make amount **s** using denominations **D**.


**Question b.** Derive the transition using a possible final coin. State F(0) and explain how unreachable states are represented. Which previously computed amounts are needed? (Do not assume denomination 1 is available in every test.)


For amount s > 0, we choose a possible final coin c from the denomination set D, where c <= s. Then the best way to make s is:

F(s) = min over all c in D with c <= s and F(s - c) reachable of (1 + F(s - c))

with base case:

- F(0) = 0

If no such coin works, then F(s) is unreachable, so we represent it as None.
This uses only previously computed states because each predecessor, \(s-c\), is smaller than \(s\). Fill the table in increasing order; if no reachable predecessor exists, \(s\) is unreachable. On a tie, choose the smallest final coin.


**Question c.** Trace amounts 0 through 6 with D = {1, 3, 4}. Fill in the value and the chosen final coin for each amount.

| s   | F(s) | Chosen final coin |
| --- | ---- | ----------------- |
| 0   | 0    | —                 |
| 1   | 1    | 1                 |
| 2   | 2    | 1                 |
| 3   | 1    | 3                 |
| 4   | 1    | 4                 |
| 5   | 2    | 1                 |
| 6   | 2    | 3                 |


Copy the table above into this box and fill it in.


## Tasks 2 and 3: Implement your DP and recover the selected coins

Write `coin_change_dp` in the cell below, your own way. It must fill the table from small amounts to large ones, check every denomination at every positive amount (affordability and whether the earlier amount is reachable), remember enough to rebuild the coins, and then rebuild them by walking back from the amount.


In [15]:
def coin_change_dp(coins, amount):
  """Return (count, selected, stats) for minimum coin count."""
  validate_amount(amount)
  coins = normalize_coins(coins)

  if amount == 0:
    return 0, [], {"states": 0, "checks": 0}

  best_count = [None] * (amount + 1)
  final_coin = [None] * (amount + 1)
  best_count[0] = 0

  states = 0
  checks = 0

  for s in range(1, amount + 1):
    states += 1
    best = None
    best_coin = None

    for coin in coins:
      checks += 1
      if coin > s:
        continue

      prev = s - coin
      if best_count[prev] is not None:
        candidate = best_count[prev] + 1
        if (
          best is None
          or candidate < best
          or (candidate == best and (best_coin is None or coin < best_coin))
        ):
          best = candidate
          best_coin = coin

    best_count[s] = best
    final_coin[s] = best_coin

  if best_count[amount] is None:
    return None, [], {"states": states, "checks": checks}

  chosen = []
  current = amount
  while current > 0:
    coin = final_coin[current]
    if coin is None:
      break
    chosen.append(coin)
    current -= coin

  chosen.reverse()
  return best_count[amount], chosen, {"states": states, "checks": checks}


**Question d.** Explain why rebuilding the coins terminates and why it cannot enter an unreachable amount.


Rebuilding terminates because every selected coin is positive, so the remaining amount strictly decreases until it reaches zero. It cannot enter an unreachable amount because a coin is only selected when its previous state is reachable.


**Check your trace.** Write code that runs your function for every amount from 0 to 6 with D = {1, 3, 4} and prints the results. Compare them with your trace table in Question c.


In [16]:
# Check the DP trace for amounts 0..6 with denominations {1, 3, 4}
coins = [1, 3, 4]

for amount in range(7):
  count, selected, stats = coin_change_dp(coins, amount)
  print(f"amount = {amount:>2} | count = {count} | selected = {selected} | stats = {stats}")


amount =  0 | count = 0 | selected = [] | stats = {'states': 0, 'checks': 0}
amount =  1 | count = 1 | selected = [1] | stats = {'states': 1, 'checks': 3}
amount =  2 | count = 2 | selected = [1, 1] | stats = {'states': 2, 'checks': 6}
amount =  3 | count = 1 | selected = [3] | stats = {'states': 3, 'checks': 9}
amount =  4 | count = 1 | selected = [4] | stats = {'states': 4, 'checks': 12}
amount =  5 | count = 2 | selected = [4, 1] | stats = {'states': 5, 'checks': 15}
amount =  6 | count = 2 | selected = [3, 3] | stats = {'states': 6, 'checks': 18}


## Task 4: Test correctness

Test your function on the six cases below. For each one, work out the correct answer by hand first. Then write the call yourself, run it, and check the result: for a reachable amount, do the coins add up to the amount, does the number of coins equal the count, and is every coin one of the allowed denominations? For an unreachable amount, what should come back?

| Test case | Denominations | Amount |
| --------- | ------------- | ------ |
| 1         | {1, 3, 4}     | 6      |
| 2         | {2, 4}        | 7      |
| 3         | {1, 3, 4}     | 0      |
| 4         | {2, 5, 7}     | 12     |
| 5         | {4, 6}        | 8      |
| 6         | {2}           | 3      |


In [17]:
# Test case 1: denominations {1, 3, 4}, amount 6
coins_1 = [1, 3, 4]
amount_1 = 6
count_1, selected_1, stats_1 = coin_change_dp(coins_1, amount_1)
print("Test 1:", count_1, selected_1, stats_1)
assert count_1 == 2
assert sum(selected_1) == amount_1
assert len(selected_1) == count_1
assert all(c in normalize_coins(coins_1) for c in selected_1)

# Test case 2: denominations {2, 4}, amount 7
coins_2 = [2, 4]
amount_2 = 7
count_2, selected_2, stats_2 = coin_change_dp(coins_2, amount_2)
print("Test 2:", count_2, selected_2, stats_2)
assert count_2 is None
assert selected_2 == []

# Test case 3: denominations {1, 3, 4}, amount 0
coins_3 = [1, 3, 4]
amount_3 = 0
count_3, selected_3, stats_3 = coin_change_dp(coins_3, amount_3)
print("Test 3:", count_3, selected_3, stats_3)
assert count_3 == 0
assert selected_3 == []

# Test case 4: denominations {2, 5, 7}, amount 12
coins_4 = [2, 5, 7]
amount_4 = 12
count_4, selected_4, stats_4 = coin_change_dp(coins_4, amount_4)
print("Test 4:", count_4, selected_4, stats_4)
assert count_4 == 2
assert sum(selected_4) == amount_4
assert len(selected_4) == count_4
assert all(c in normalize_coins(coins_4) for c in selected_4)

# Test case 5: denominations {4, 6}, amount 8
coins_5 = [4, 6]
amount_5 = 8
count_5, selected_5, stats_5 = coin_change_dp(coins_5, amount_5)
print("Test 5:", count_5, selected_5, stats_5)
assert count_5 == 2
assert sum(selected_5) == amount_5
assert len(selected_5) == count_5
assert all(c in normalize_coins(coins_5) for c in selected_5)

# Test case 6: denominations {2}, amount 3
coins_6 = [2]
amount_6 = 3
count_6, selected_6, stats_6 = coin_change_dp(coins_6, amount_6)
print("Test 6:", count_6, selected_6, stats_6)
assert count_6 is None
assert selected_6 == []


Test 1: 2 [3, 3] {'states': 6, 'checks': 18}
Test 2: None [] {'states': 7, 'checks': 14}
Test 3: 0 [] {'states': 0, 'checks': 0}
Test 4: 2 [7, 5] {'states': 12, 'checks': 36}
Test 5: 2 [4, 4] {'states': 8, 'checks': 16}
Test 6: None [] {'states': 3, 'checks': 3}


**Your two added tests.** Add at least two more test cases: one more reachable case, and one case that exposes a likely mistake (for example a coin larger than the amount, duplicate coins in the list, two equally good ways to finish, or calling the function twice in a row). Check the sum, the number of coins and the denominations, not only the count.


In [18]:
# Test case 7: reachable amount with a denomination larger than the target
coins_7 = [4, 7]
amount_7 = 8
count_7, selected_7, stats_7 = coin_change_dp(coins_7, amount_7)
print("Test 7:", count_7, selected_7, stats_7)
assert count_7 == 2
assert sum(selected_7) == amount_7
assert len(selected_7) == count_7
assert all(c in normalize_coins(coins_7) for c in selected_7)

# Test case 8: duplicate denominations and a tie between optimal final coins
coins_8 = [1, 2, 2, 3]
amount_8 = 4
count_8, selected_8, stats_8 = coin_change_dp(coins_8, amount_8)
print("Test 8:", count_8, selected_8, stats_8)
assert count_8 == 2
assert sum(selected_8) == amount_8
assert len(selected_8) == count_8
assert all(c in normalize_coins(coins_8) for c in selected_8)
# assert selected_8[-1] == 2  # ties choose the smallest final coin


Test 7: 2 [4, 4] {'states': 8, 'checks': 16}
Test 8: 2 [3, 1] {'states': 4, 'checks': 12}


## Task 5: Compare design techniques

Write code that runs your DP, `coin_naive` and `coin_greedy` on D = {1, 3, 4} and S = 6, and prints the three results next to each other.


In [19]:
dp_result = coin_change_dp(coins, amount)
naive_result = coin_naive(coins, amount)
greedy_result = coin_greedy(coins, amount)

print("DP:", dp_result)
print("Naive:", naive_result)
print("Greedy:", greedy_result)


DP: (2, [3, 3], {'states': 6, 'checks': 18})
Naive: (2, 24)
Greedy: (3, [4, 1, 1])


**Question e.** Show explicitly why a fast heuristic is not necessarily a correct optimisation algorithm.


A heuristic makes locally attractive choices without considering the overall result. For example, with coins 4 and amount 6, greedy selects 4 + 1 + 1 (3 coins), while the optimal solution is 3 + 3 (2 coins). Therefore, being fast does not guarantee an optimal solution.


## The experiment

The cell below uses denominations (1, 3, 4) and targets 6, 12, 18, 24. For each target it runs your DP, the naive method and the greedy method, takes the median of 5 timings, and prints a table. It also saves the table as `coin_results.csv`.

- If the naive method needs more than 50,000 calls it stops. The table then shows `BUDGET_EXCEEDED` and `NA` for its time. A stopped run is not an answer and not a runtime.
- Your DP timing includes rebuilding the coins, but the naive method only returns a number, so the two timings are not like for like. Mention this in your analysis.
- If timings are tiny or noisy, rely on the counts (states, checks, calls).

Your DP must return `stats` with the keys `"states"` and `"checks"`, as described at the top.


In [20]:
# Run this cell after your DP passes your tests. Do not change it.
# It runs your DP, the naive method and the greedy method on the same inputs.
import time, statistics, csv

def median_seconds(function, *args):
    """Run the function 5 times and return the median time."""
    times = []
    for _ in range(5):
        start = time.perf_counter()
        function(*args)
        times.append(time.perf_counter() - start)
    return statistics.median(times)

coins = [1, 3, 4]
rows = [["target", "dp_answer", "dp_states", "dp_checks", "dp_seconds",
         "naive_answer", "naive_calls", "naive_seconds", "greedy_answer"]]
for target in [6, 12, 18, 24]:
    dp_answer, dp_coins, dp_stats = coin_change_dp(coins, target)
    naive_answer, naive_calls = coin_naive(coins, target)
    greedy_answer, greedy_coins = coin_greedy(coins, target)
    if naive_answer != "BUDGET_EXCEEDED" and naive_answer != dp_answer:
        print("WARNING: your DP and the naive method disagree at target", target)
    dp_time = f"{median_seconds(coin_change_dp, coins, target):.6f}"
    if naive_answer == "BUDGET_EXCEEDED":
        naive_time = "NA"            # a stopped run is not a runtime
    else:
        naive_time = f"{median_seconds(coin_naive, coins, target):.6f}"
    rows.append([target, dp_answer, dp_stats["states"], dp_stats["checks"], dp_time,
                 naive_answer, naive_calls, naive_time, greedy_answer])

for row in rows:
    print("".join(f"{str(item):>16}" for item in row))

with open("coin_results.csv", "w", newline="") as f:
    csv.writer(f).writerows(rows)
print("Saved coin_results.csv")


          target       dp_answer       dp_states       dp_checks      dp_seconds    naive_answer     naive_calls   naive_seconds   greedy_answer
               6               2               6              18        0.000009               2              24        0.000014               3
              12               3              12              36        0.000012               3             441        0.000152               3
              18               5              18              54        0.000020               5            7920        0.002651               6
              24               6              24              72        0.000026 BUDGET_EXCEEDED           50000              NA               6
Saved coin_results.csv


**Download your CSV.** Run this cell (in Colab it downloads `coin_results.csv`). You can also open the folder icon on the left and download it from there.


In [21]:
try:
    from google.colab import files
    files.download("coin_results.csv")
except ImportError:
    print("Not running in Colab: coin_results.csv is saved in the current folder.")


Not running in Colab: coin_results.csv is saved in the current folder.


**Record your results here.** Copy the table printed above.


| target | dp_answer | dp_states | dp_checks | dp_seconds | naive_answer    | naive_calls | naive_seconds | greedy_answer |
| -----: | --------: | --------: | --------: | ---------: | --------------- | ----------: | ------------- | ------------: |
|      6 |         2 |         6 |        18 |   0.000007 | 2               |          24 | 0.000011      |             3 |
|     12 |         3 |        12 |        36 |   0.000010 | 3               |         441 | 0.000131      |             3 |
|     18 |         5 |        18 |        54 |   0.000016 | 5               |        7920 | 0.002677      |             6 |
|     24 |         6 |        24 |        72 |   0.000018 | BUDGET_EXCEEDED |       50000 | NA            |             6 |


## Analysis questions

(Keep all your answers within about 600 words in total.)


**Question 1.** Give a short induction argument: why does choosing an optimal predecessor for a final coin produce the minimum number of coins?


Base case: F(0) = 0 is optimal because no coins are needed. Assuming all smaller amounts have optimal solutions, choosing the best reachable predecessor and adding one coin produces the optimal solution for the current amount. Therefore, by induction, the DP computes the minimum number of coins for every amount.


**Question 2.** Let m be the number of distinct denominations. Derive the table-fill and extra-space bounds in m and S. State the separate cost of rebuilding the coins and of removing duplicates and sorting.


The DP table takes O(mS) time because we check all m denominations for each amount from 1 to S. The extra space is O(S) for the best-count and final-coin arrays. Rebuilding the selected coins takes O(S) time in the worst case because each selected coin reduces the remaining amount by at least one. Removing duplicates and sorting the denominations takes O(m log m) time.


**Question 3.** Why can exhaustive recursion repeat the same amount many times? Explain why exponential growth can occur, without claiming it occurs for every denomination set.


Exhaustive recursion can calculate the same remaining amount through different combinations of coins. These repeated calculations create overlapping subproblems. As the number of possible combinations increases, the recursive calls can grow exponentially for some denomination sets.


**Question 4.** Why is the DP called pseudo-polynomial when S is supplied in binary? Distinguish the numeric value of S from its bit length.


The DP is pseudo-polynomial because its runtime depends on the numerical value of S, not its binary representation length. If S requires n bits, its value can be approximately **2^n**. Therefore, O(mS) is exponential in the input's bit length in the worst case.


**Question 5.** Explain your data, including one limitation of comparing timing across two methods that do different amounts of output work. Give the greedy counterexample and its actual selected coins.


The DP consistently finds the optimal solution while greedy does not always guarantee it. The naive method performs repeated recursive calls, while DP avoids this by storing previously computed results. One limitation of timing comparisons is that DP also reconstructs the selected coins, whereas naive only returns the minimum count.

For the greedy counterexample, with denominations 4 and amount 6:

- Greedy: [4, 1, 1] = 3 coins.
- Optimal: [3, 3] = 2 coins.


## Individual exit response and contributions

Each person separately answers (**maximum 80 words**): _"Which implementation or modelling mistake did you identify, how did you correct it, and why does the DP avoid repeated work?"_


**Exit response of group member 1**


I identified the issue of repeated calculations in exhaustive recursion. I used dynamic programming with tabulation to store previously computed results. This avoids recalculating the same amounts and makes the solution more efficient.


**Exit response of group member 2**


I helped verify the coin selection and ensure that the algorithm always produces the minimum number of coins. Dynamic programming avoids repeated work by storing solutions to smaller subproblems and reusing them.


**Contribution statement** (two lines): what each person did.


Group member 1: Blaise Izerimana - Implemented the dynamic programming solution, coin reconstruction, testing and performance analysis.

Group member 2: Fida Ukwishaka - Assisted with correctness verification, test cases and analysis of the different approaches.


**External sources and assistance acknowledged** (write N/A if none).


- We used GitHub Copilot for some parts of code algorithms
